In [ ]:
import pandas as pd
from pathlib import Path
import re
from plots import plot_histogram, plot_rank_boxplot
from evaluation import validate_distances

xp_name = "compare_algorithm"
xp_path = Path(f"results/{xp_name}")

In [ ]:
# Discover experiment structure: {xp_base_path}/{xp_name}/{graph}/{query_hash}/{device_id}/*.csv
experiments = {}

# Scan through the hierarchy
for graph_dir in xp_path.iterdir():
    if not graph_dir.is_dir():
        continue
    graph_name = graph_dir.name
    
    for query_hash_dir in graph_dir.iterdir():
        if not query_hash_dir.is_dir():
            continue
        query_hash = query_hash_dir.name
        
        for device_id_dir in query_hash_dir.iterdir():
            if not device_id_dir.is_dir():
                continue
            device_id = device_id_dir.name
            
            exp_key = (graph_name, query_hash, device_id)
            if exp_key not in experiments:
                experiments[exp_key] = {}
            
            for csv_file in device_id_dir.glob('*.csv'):
                # Parse filename: {timestamp}_{gitsha}_{variant}.csv
                match = re.match(r'(\d+)_([^_]+)_(.+)\.csv', csv_file.name)
                if not match:
                    continue
                
                timestamp = int(match.group(1))
                git_sha = match.group(2)
                variant = match.group(3)

                if variant not in experiments[exp_key]:
                    experiments[exp_key][variant] = []
                experiments[exp_key][variant].append((timestamp, csv_file))

# Sort by timestamp (most recent first) and take the most recent for each algorithm
for exp_key in experiments:
    for variant in experiments[exp_key]:
        experiments[exp_key][variant].sort(reverse=True, key=lambda x: x[0])

print(f"Found {len(experiments)} experiment configurations:")
for exp_key, data in experiments.items():
    graph_name, query_hash, device_id = exp_key
    print(f"\nGraph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    for variant in experiments[exp_key]:
        print(f"  {variant}: {len(data[variant])} files")

In [ ]:
# Load data for each complete experiment set
experiment_data = {}

for exp_key, variants in experiments.items():
    graph_name, query_hash, device_id = exp_key

    variant_dfs = {variant: pd.read_csv(files[0][1]) for variant, files in variants.items()}
    
    print(f"\nGraph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    for variant, df in variant_dfs.items():
        print(f"  {variant}: {len(df)} queries")

    all_df = None
    for var, df in variant_dfs.items():
        if all_df is None:
            all_df = df.rename(columns={'distance': f'distance_{var}', 'time': f'time_{var}'})
        else:
            all_df = pd.merge(
                all_df,
                df,
                on=['from_node_id', 'to_node_id', 'rank']
            )
            all_df.rename(columns={'distance': f'distance_{var}', 'time': f'time_{var}'}, inplace=True)
    
    experiment_data[exp_key] = variant_dfs
    print(f"  Merged: {len(all_df)} queries")

In [ ]:
# Check for distance errors for each experiment set
for exp_key, variant_dfs in experiment_data.items():
    graph_name, query_hash, device_id = exp_key

    if device_id == "cpu":
        continue
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)

    baseline_dfs = experiment_data.get((graph_name, query_hash, "cpu"), {})
    validate_distances(baseline_dfs | variant_dfs)

In [ ]:
# Histogram comparing execution times for each experiment set
for exp_key, variant_dfs in experiment_data.items():
    graph_name, query_hash, device_id = exp_key

    if device_id == "cpu":
        continue
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)

    baseline_dfs = experiment_data.get((graph_name, query_hash, "cpu"), {})
    plot_histogram(baseline_dfs | variant_dfs, f"{graph_name} device={device_id}")

In [ ]:
# Box plot by query rank for each experiment set
for exp_key, variant_dfs in experiment_data.items():
    graph_name, query_hash, device_id = exp_key

    if device_id == "cpu":
        continue
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)

    baseline_dfs = experiment_data.get((graph_name, query_hash, "cpu"), {})
    plot_rank_boxplot(baseline_dfs | variant_dfs, f"{graph_name} device={device_id}")